## Welcome to the Second Lab - Week 1, Day 3

Today we will work with lots of models! This is a way to get comfortable with APIs.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important point - please read</h2>
            <span style="color:#ff7800;">The way I collaborate with you may be different to other courses you've taken. I prefer not to type code while you watch. Rather, I execute Jupyter Labs, like this, and give you an intuition for what's going on. My suggestion is that you carefully execute this yourself, <b>after</b> watching the lecture. Add print statements to understand what's going on, and then come up with your own variations.<br/><br/>If you have time, I'd love it if you submit a PR for changes in the community_contributions folder - instructions in the resources. Also, if you have a Github account, use this to showcase your variations. Not only is this essential practice, but it demonstrates your skills to others, including perhaps future clients or employers...
            </span>
        </td>
    </tr>
</table>

In [18]:
# Start with imports - ask ChatGPT to explain any package that you don't know

import os
import json
from dotenv import load_dotenv
from groq import Groq
from IPython.display import Markdown, display
from openai import OpenAI

In [19]:
# Always remember to do this!
load_dotenv(override=True)

True

In [20]:
# Print the key prefixes to help with any debugging

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

OpenAI API Key not set
Anthropic API Key not set (and this is optional)
Google API Key not set (and this is optional)
DeepSeek API Key not set (and this is optional)
Groq API Key exists and begins gsk_


In [21]:
request = "Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence. "
request += "Answer only with the question, no explanation."
messages = [{"role": "user", "content": request}]

In [22]:
messages

[{'role': 'user',
  'content': 'Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence. Answer only with the question, no explanation.'}]

In [23]:
groq = Groq()
response = groq.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=messages,
)
question = response.choices[0].message.content
print(question)


If you are designing a recommendation system that must personalize content for millions of users while ensuring fairness across protected demographic groups and maintaining high engagement metrics, how would you formulate a multi-objective optimization problem that balances personalization accuracy, demographic parity, and user retention, and what algorithmic approach would you use to solve it efficiently in a streaming data setting?


In [24]:
competitors = []
answers = []
messages = [{"role": "user", "content": question}]

## Note - update since the videos

I've updated the model names to use the latest models below, like GPT 5 and Claude Sonnet 4.5. It's worth noting that these models can be quite slow - like 1-2 minutes - but they do a great job! Feel free to switch them for faster models if you'd prefer, like the ones I use in the video.

In [25]:
# The API we know well
# I've updated this with the latest model, but it can take some time because it likes to think!
# Replace the model with gpt-4.1-mini if you'd prefer not to wait 1-2 mins

model_name = "openai/gpt-oss-20b"

response = groq.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

Below is a compact yet complete recipe you can drop into a production notebook, a slide deck, or a research paper.

---

## 1.  What you’re trying to optimise

| Target | What it means for a “good” recommender | Typical loss/metric |
|--------|----------------------------------------|---------------------|
| **Personalisation accuracy** | The model’s predictions should be close to a user’s true preference. | `ℓ_acc(θ)` – e.g. cross‑entropy or BPR‑rank loss. |
| **Fairness (demographic parity)** | The *average* probability of showing a recommended item should be the same for all protected groups. | `Δ_grp(θ) = |E[p | A=0] – E[p | A=1]|`. |
| **User retention** | Long‑term engagement (e.g. probability a user remains active 30 days later). | `ℓ_ret(θ)` – a logistic regression on a retention label, or a survival‑analysis loss. |

All three objectives are **mutually compatible but often conflict** (e.g. a pure accuracy‑maximising model can give a higher hit‑rate to one group).  
The goal is therefore a *Pareto‑optimal* point that balances them according to business priorities.

---

## 2.  Formal Multi‑Objective Optimization

Let  

* `θ ∈ ℝ^d` – global model parameters (e.g. latent‑factor matrix, neural‑network weights).  
* `x_i` – feature vector for user *i* (context, history, device, etc.).  
* `a_i ∈ {0,1}` – protected‑group label (0 = unprotected, 1 = protected).  
* `y_i ∈ {0,1}` – observed preference (click, view, buy).  
* `p_i(θ) = σ(θᵀx_i)` – predicted probability of a positive outcome.  
* `R_i(t)` – indicator that user *i* is still active at day *t* (retention label).

### 2.1  Objective: weighted‑sum scalarisation

```math
min_{θ,λ₁,λ₂}  
   L(θ,λ) =  \underbrace{L_acc(θ)}_{\text{accuracy}}
           + λ₁  \underbrace{L_ret(θ)}_{\text{retention}}
           + λ₂  \underbrace{L_fair(θ)}_{\text{fairness}}
```

* `L_acc(θ)` – e.g. negative log‑likelihood over the current minibatch.  
* `L_ret(θ)` – expected log‑loss of a simple logistic regressor that predicts `R_i(t)` from `p_i(θ)`.  
* `L_fair(θ)` – a *penalty* that encourages demographic parity:

```math
L_fair(θ)=   \max\{ 0,  | \bar{p}_0 - \bar{p}_1 | - ε  \}
```

where  

* `\bar{p}_g = 1/|S_g| ∑_{i∈S_g} p_i(θ)`  (average predicted score for group *g*)  
* `S_g` – the set of users in group *g* seen in the current window.  
* `ε` – a small tolerance that lets us trade a bit of parity for better accuracy / retention.

### 2.2  Constraints (optional)

If you prefer hard constraints rather than a penalty, replace the penalty by

```math
| \bar{p}_0 - \bar{p}_1 | ≤ ε
```

and add a Lagrange multiplier `λ₂ ≥ 0`. The same idea applies to a retention target `R̄ ≥ R_min`.

---

## 3.  Streaming‑Friendly Algorithmic Blueprint

Because the data arrive online and the user base is in the millions, you need an **O(d)** per‑sample algorithm that can run asynchronously on many machines. The classic recipe is an **online primal‑dual (primal‑gradient + dual‑subgradient) scheme**.

### 3.1  Data structures

| Item | Why it’s needed | Implementation tips |
|------|-----------------|---------------------|
| **Model parameters `θ`** | Global model | Stored in a parameter‑server (Sharded by hash(user)). |
| **Per‑group moving averages** `\bar{p}_0, \bar{p}_1` | Fairness statistics | Maintain exponential moving average (EMA) with decay α≈0.99. |
| **Retention model** (logistic regression on `p`) | Predicts long‑term engagement | Updated with its own learning rate, or use the same `θ` if you embed retention features. |
| **Dual variables** `λ₁, λ₂` | Enforce fairness / retention | Scalars updated with subgradient steps. |

All of these are tiny compared to `θ`, so they can stay in RAM on a single node.

### 3.2  Online update loop (per event)



In [29]:
groq = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")
response = groq.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=messages,
)
question = response.choices[0].message.content
print(question)

## 1.  Problem framing

We want a **streaming recommendation engine** that serves *millions* of users, each time step \(t\) we observe a user–context pair \((u_t, x_t)\), choose an item \(a_t\), see the feedback \(y_t \in \{0,1\}\) (e.g. click, view, purchase) and an *engagement* signal (e.g. dwell‑time, return‑rate).  
The system must

| Goal | How it is measured |
|------|--------------------|
| **Personalization accuracy** | Prediction error on the user‑specific preference model (e.g. negative log‑likelihood, squared error). |
| **Fairness / demographic parity** | The probability that a protected group \(g \in \mathcal{G}\) receives a positive recommendation should be the same (or within a tolerance \(\epsilon\)). |
| **User retention / engagement** | Long‑term reward (e.g. cumulative click‑through‑rate, session length, churn probability). |

We need a single **multi‑objective optimisation** that balances all three simultaneously and can be updated on the fly as new data arrives.

-

In [26]:
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
model_name = "mistral:instruct"

response = ollama.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

 To design a recommendation system that personalizes content for millions of users while ensuring fairness, high engagement, and personalization accuracy, we can formulate a multi-objective optimization problem (MOP) with the following objectives:

1. Personalization Accuracy (f1): Measure the accuracy of recommendations based on user feedback (clicks, likes, ratings, etc.). Higher f1 scores reflect better personalization accuracy.

2. Demographic Parity (eq): Ensuring that recommendations are equally accessible and beneficial to all demographic groups (e.g., gender, ethnicity, age, etc.) to avoid biases and promote fairness.

3. User Retention (ur): Measure the ability of the recommendations to keep users engaged to maintain their long-term usage of the system.

In order to formulate this MOP, we need a suitable aggregation method for the demographic parity objective (eq) as it consists of multiple demographic variables. One common practice is to use one metric for each demographic variable and aggregate them using a weighted average or a composite metric like the combined disparate impact (CDI).

Our MOP can be formulated as:

Minimize F(x) = (1 - ρ)*f1(x) + ρ*eq(x) + (1 - ω)*ur(x)

where ρ and ω are weights assigned to the demographic parity and user retention objectives, respectively (the sum of weights should be less than 1).

To solve this real-time, resource-efficient, and fair streaming MOP, we can utilize a Multi-Objective Evolutionary Algorithm (MOEA) that can explore the large solution space efficiently and adaptively find promising solutions. A streaming MOEA variant such as Streaming-MOEA or Nondominated Sorting Distance (NSD) MOEA can be appropriate for handling high-dimensional data streams.

These algorithms work by iteratively updating a population of candidate solutions using a variety of operators like selection, recombination, and mutation. These operators encourage exploration (discovering diverse solutions) and exploitation (exploring promising solutions) to maintain an diverse and high-quality set of solutions throughout the process.

In this context, the recommendation system can be implemented using the identified solutions while continually adapting to the streaming data and user preferences, thereby allowing it to balance personalization accuracy, demographic parity, and user retention.

In [30]:
# Updated with the latest Open Source model from OpenAI

groq = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")
model_name = "openai/gpt-oss-120b"

response = groq.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)


The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.


Below is a **complete recipe** for turning the business goal *“personalize for each of millions of users while keeping the system fair and sticky”* into a **formal multi‑objective optimization problem** and then solving it **online** as new interaction events stream in.

---

## 1.  Problem statement in words  

| Goal | Symbolic name | What we measure |
|------|---------------|-----------------|
| **Personalization accuracy** | \( \mathcal{A} \) | Click‑through‑rate (CTR) or rating‑prediction loss on the *individual* level. |
| **Demographic parity** (group fairness) | \( \mathcal{F} \) | Difference (or KL‑divergence) between the distribution of recommended items (or exposure) for each protected group \(g\in\mathcal{G}\) and the overall distribution. |
| **User retention** | \( \mathcal{R} \) | Probability that a user stays active after the recommendation (e.g., 7‑day return rate, churn‑prediction loss). |

All three have to be **optimized simultaneously**: we want the best possible \(\mathcal{A}\) *subject* to a bounded fairness violation and we also want to keep \(\mathcal{R}\) high because a sticky user gives us more data for future personalization.

---

## 2.  Formal multi‑objective formulation  

### 2.1  Notation  

* \(U\) – set of users (size \(10^6\)‑\(10^9\)).  
* \(I\) – set of items (catalog).  
* \(\mathbf{x}_{u,i} \in \mathbb{R}^d\) – feature vector for user‑item pair (user embedding, item embedding, context).  
* \(\theta \in \mathbb{R}^p\) – model parameters (e.g., weights of a neural‑ranking function \(f_\theta(\mathbf{x}_{u,i})\)).  
* \(y_{u,i} \in \{0,1\}\) – observed engagement (click / conversion).  
* \(g(u) \in \mathcal{G}\) – protected‑group identifier for user \(u\) (e.g., gender, ethnicity).  

We will treat the **recommendation list** for a user at time \(t\) as the top‑\(K\) items according to the score  
\[
\hat{s}_{u,i}(t) = f_\theta\bigl(\mathbf{x}_{u,i}(t)\bigr).
\]

### 2.2  Three loss components  

1. **Personalization loss** (binary cross‑entropy, negative log‑likelihood, or pairwise ranking loss)  
   \[
   \mathcal{L}_{\text{acc}}(\theta) = \frac{1}{| \mathcal{D}_t |}\sum_{(u,i,y)\in\mathcal{D}_t}\ell_{\text{acc}}\!\bigl(y,\,\sigma(\hat{s}_{u,i}(t))\bigr)
   \]
   where \(\mathcal{D}_t\) is the minibatch of events observed up to time \(t\) and \(\sigma\) is the sigmoid.

2. **Demographic‑parity regularizer** – we compare the **exposure** each group receives to the global exposure.  
   *Exposure* for group \(g\) on a single recommendation slate is the sum of indicator that an item from the slate is shown:  
   \[
   \operatorname{Exp}_g(t) = \frac{1}{|U_g|}\sum_{u\in U_g}\!\!\sum_{i\in \text{TopK}_u(t)}\frac{1}{K}.
   \]  
   The **fairness violation** (e.g., absolute difference) is  
   \[
   \mathcal{L}_{\text{fair}}(\theta) = \sum_{g\in\mathcal{G}} \bigl| \operatorname{Exp}_g(t) - \operatorname{Exp}_{\text{all}}(t) \bigr|.
   \]  
   (Other choices: squared difference, KL‑divergence, or the *disparate impact ratio*.)

3. **Retention loss** – we learn a *churn predictor* \(r_\phi\) (tiny auxiliary model) that estimates the probability the user will return after the slate.  
   \[
   \mathcal{L}_{\text{ret}}(\theta,\phi) = \frac{1}{|U_t|}\sum_{u\in U_t}\ell_{\text{ret}}\!\bigl( \underbrace{z_u}_{\text{observed 7‑day return}},\, r_\phi(\mathbf{c}_u(t)) \bigr)
   \]  
   where \(\mathbf{c}_u(t)\) are context features (e.g., current slate scores, past activity). The *gradient* of \(\mathcal{L}_{\text{ret}}\) with respect to \(\theta\) flows through the slate scores because the churn predictor uses them as input.

### 2.3  Scalarised objective (weighted sum)  

A classic way to obtain a **single‑objective** amenable to stochastic optimisation is to introduce *trade‑off weights* \(\alpha,\beta,\gamma\ge 0\):  

\[
\boxed{
\min_{\theta,\phi}\; \underbrace{\alpha\,\mathcal{L}_{\text{acc}}(\theta)}_{\text{accuracy}}
\;+\; \underbrace{\beta\,\mathcal{L}_{\text{fair}}(\theta)}_{\text{fairness}}
\;+\; \underbrace{\gamma\,\mathcal{L}_{\text{ret}}(\theta,\phi)}_{\text{retention}} 
\;+\; \lambda \|\theta\|_2^2
}
\tag{1}
\]

* \(\alpha\) reflects the business priority on click quality.  
* \(\beta\) penalises any deviation from demographic parity.  
* \(\gamma\) rewards slates that keep users coming back.  
* \(\lambda\) is a standard L2 regulariser for stability.

### 2.4  Constrained version (for strict fairness)  

If the product team demands a **hard bound** on parity, we can move \(\mathcal{L}_{\text{fair}}\) into a constraint:  

\[
\begin{aligned}
\min_{\theta,\phi}\;& \alpha\,\mathcal{L}_{\text{acc}}(\theta) + \gamma\,\mathcal{L}_{\text{ret}}(\theta,\phi) + \lambda\|\theta\|_2^2 \\[4pt]
\text{s.t. } &\mathcal{L}_{\text{fair}}(\theta) \le \epsilon .
\end{aligned}
\tag{2}
\]

Both (1) and (2) are *convex* in \(\theta\) only if we use a linear model and convex losses; with deep nets they become non‑convex but still tractable for stochastic gradient methods.

---

## 3.  Why a *multi‑objective* formulation matters  

* **Pareto front awareness** – By exposing the three loss terms we can trace the trade‑off curve (e.g., how much accuracy we lose for a 1 % fairness improvement). This is essential for product‑leadership negotiations.  
* **Dynamic weighting** – In a streaming environment the relative importance of \(\alpha,\beta,\gamma\) can be *adapted* on‑the‑fly (e.g., increase \(\beta\) when a fairness audit flag is triggered, increase \(\gamma\) after a sudden churn spike).

---

## 4.  Algorithmic solution in a **streaming / online** setting  

### 4.1  High‑level requirements  

| Requirement | How we satisfy it |
|-------------|-------------------|
| **One‑pass, low‑latency updates** | Use **online stochastic gradient descent (SGD)** / **Follow‑The‑Regularized‑Leader (FTRL)** that processes each event as it arrives. |
| **Memory‑efficient fairness statistics** | Maintain *exponential moving averages* (EMA) for group‑wise exposure and overall exposure, rather than recomputing from scratch. |
| **Scalable to millions of users** | Parameter server / shard the embedding tables; the gradient update touches only the active user and the candidate items. |
| **Guaranteed constraint satisfaction (optional)** | Apply a **primal‑dual (Lagrangian) online method** that updates a fairness multiplier \(\mu_t\) each step, projecting the multiplier onto \([0,\infty)\). |
| **Non‑stationary data (concept drift)** | EMA’s decay factor \(\eta\) lets the system forget old behaviour; we can also add a small *regularisation drift* term. |

### 4.2  Core algorithm – *Primal‑Dual Online Mirror Descent*  

We treat the scalarised objective (1) *or* the constrained version (2) with a **Lagrangian**:

\[
\mathcal{L}_t(\theta,\phi,\mu) = \alpha\,\ell_{\text{acc}}^{(t)}(\theta) 
+ \gamma\,\ell_{\text{ret}}^{(t)}(\theta,\phi) 
+ \mu\bigl(\ell_{\text{fair}}^{(t)}(\theta)-\epsilon\bigr)
+ \lambda\|\theta\|_2^2 .
\]

At each event \(t\) (or small mini‑batch) we perform:

1. **Gradient step for primal variables**  
   \[
   \theta_{t+1} = \Pi_{\Theta}\!\Bigl(\theta_t - \eta_t \nabla_\theta \mathcal{L}_t(\theta_t,\phi_t,\mu_t)\Bigr)
   \]  
   \[
   \phi_{t+1} = \Pi_{\Phi}\!\Bigl(\phi_t - \eta_t \nabla_\phi \bigl(\gamma\,\ell_{\text{ret}}^{(t)}(\theta_t,\phi_t)\bigr)\Bigr)
   \]  
   where \(\Pi\) denotes projection onto a feasible set (e.g., box constraints to keep embeddings bounded).  

2. **Dual ascent for fairness multiplier**  
   \[
   \mu_{t+1} = \bigl[ \mu_t + \eta_t^{\mu}\,(\ell_{\text{fair}}^{(t)}(\theta_t)-\epsilon) \bigr]_+ .
   \]  
   The ``\([\,\cdot\,]_+``) is the ReLU projection onto \(\mathbb{R}_{\ge0}\).  

3. **Update EMA of exposure** (used inside \(\ell_{\text{fair}}^{(t)}\))  
   \[
   \operatorname{Exp}_g \leftarrow (1-\rho)\operatorname{Exp}_g + \rho\,\frac{1}{K}\sum_{i\in\text{TopK}_u} \mathbf{1}\{g(u)=g\}
   \]  
   with a decay \(\rho\in(0,1)\) (e.g., 0.001).  

4. **Optional: Adaptive step‑size** – Use AdaGrad / Adam‑style moments for \(\theta\) and \(\phi\), but keep a **global** learning‑rate schedule for \(\mu\) to avoid oscillations.

#### Pseudocode

```python
# hyper‑parameters
alpha, beta, gamma, epsilon = 1.0, 0.5, 0.3, 0.02
eta_primal = 1e-3
eta_dual   = 5e-4
rho_fair   = 1e-3   # EMA decay for fairness stats

# state
theta, phi = init_model()
mu        = 0.0
exp_g     = {g: 0.0

## For the next cell, we will use Ollama

Ollama runs a local web service that gives an OpenAI compatible endpoint,  
and runs models locally using high performance C++ code.

If you don't have Ollama, install it here by visiting https://ollama.com then pressing Download and following the instructions.

After it's installed, you should be able to visit here: http://localhost:11434 and see the message "Ollama is running"

You might need to restart Cursor (and maybe reboot). Then open a Terminal (control+\`) and run `ollama serve`

Useful Ollama commands (run these in the terminal, or with an exclamation mark in this notebook):

`ollama pull <model_name>` downloads a model locally  
`ollama ls` lists all the models you've downloaded  
`ollama rm <model_name>` deletes the specified model from your downloads

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Super important - ignore me at your peril!</h2>
            <span style="color:#ff7800;">The model called <b>llama3.3</b> is FAR too large for home computers - it's not intended for personal computing and will consume all your resources! Stick with the nicely sized <b>llama3.2</b> or <b>llama3.2:1b</b> and if you want larger, try llama3.1 or smaller variants of Qwen, Gemma, Phi or DeepSeek. See the <A href="https://ollama.com/models">the Ollama models page</a> for a full list of models and sizes.
            </span>
        </td>
    </tr>
</table>

In [31]:
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
model_name = "llama3"

response = ollama.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

What a fascinating question!

Formulating a multi-objective optimization problem to balance personalization accuracy, demographic parity, and user retention can be done as follows:

**Objective Functions:**

1. **Personalization Accuracy** (PA): This objective function aims to maximize the average accuracy of content recommendations for each user. PA can be measured using metrics like mean average precision (MAP) or normalized discounted cumulative gain (NDCG).
2. **Demographic Parity** (DP): This objective function ensures fairness across protected demographic groups by maximizing the average accuracy of recommendations for each demographic group. DP can be measured using metrics like demographic parity index (DPI) or equalized odds (EO).
3. **User Retention** (UR): This objective function aims to maximize the percentage of users who continue engaging with the system over time. UR can be measured using metrics like 30-day retention rate or average engagement time.

**Constraints:**

1. **Fairness constraint**: Ensure that the personalization accuracy for each demographic group is above a certain threshold (e.g., 80%).
2. **Capacity constraint**: Ensure that the system can handle the volume of user interactions and processing power requirements.
3. **Regulatory constraints**: Adhere to relevant regulations, such as GDPR or COPPA, regarding data protection and user privacy.

**Multi-Objective Optimization Formulation:**
```math
maximize PA (P1, ..., Pn), DP (P1, ..., Pn), and UR (P1, ..., Pn)
subject to:
    PA ≥ θ (fairness threshold)
    UR ≤ 1 - γ (retention rate threshold)
    P1, ..., Pn ∈ [0, 1] (probability constraint)

where P1, ..., Pn are the probability of recommending content to user i, θ is the fairness threshold, and γ is the retention rate threshold.
```
**Algorithmic Approach:**

1. **Gradient-based optimization**: Use a gradient-based optimization algorithm, such as stochastic gradient descent (SGD) or conjugate gradient (CG), to optimize the multi-objective problem.
2. **Model-based approach**: Use a probabilistic model, such as a mixture-of-experts (MoE) or a probabilistic matrix factorization (PMF) model, to represent the user-content interaction patterns.
3. ** Hierarchical clustering**: Apply hierarchical clustering to group users based on their engagement patterns, demographics, and content preferences.
4. **Batching and chunking**: Process user interactions in batches (e.g., every 1000 interactions) and chunk the data to reduce computational requirements and improve scalability.

**Streaming Data Setting:**

1. **Event-triggered optimization**: Optimize the model parameters whenever a new user interaction event occurs (e.g., a user clicks on a recommended content).
2. **Incremental learning**: Update the model parameters incrementally using online learning algorithms, such as gradient boosting (GB) or online principal component analysis (OPCA).
3. **Data augmentation**: Use techniques like data augmentation or synthetic data generation to increase the diversity of the training data and improve robustness.

**Efficiency Considerations:**

1. **Parallel processing**: Utilize shared-memory parallel processing frameworks (e.g., OpenMP) or distributed computing frameworks (e.g., Hadoop, Spark) to speed up computations.
2. **Data caching**: Implement data caching mechanisms to reduce the number of database query and improve performance.
3. **Model pruning and compression**: Apply model pruning and compression techniques to reduce the computational requirements and memory footprint.

By formulating a multi-objective optimization problem and applying the proposed algorithmic approach, you can develop a recommendation system that balances personalization accuracy, demographic parity, and user retention while efficiently handling the streaming data setting.

In [32]:
# So where are we?

print(competitors)
print(answers)


['openai/gpt-oss-20b', 'mistral:instruct', 'openai/gpt-oss-120b', 'llama3']
['Below is a compact yet complete recipe you can drop into a production notebook, a slide deck, or a research paper.\n\n---\n\n## 1.  What you’re trying to optimise\n\n| Target | What it means for a “good” recommender | Typical loss/metric |\n|--------|----------------------------------------|---------------------|\n| **Personalisation accuracy** | The model’s predictions should be close to a user’s true preference. | `ℓ_acc(θ)` – e.g. cross‑entropy or BPR‑rank loss. |\n| **Fairness (demographic parity)** | The *average* probability of showing a recommended item should be the same for all protected groups. | `Δ_grp(θ) = |E[p | A=0] – E[p | A=1]|`. |\n| **User retention** | Long‑term engagement (e.g. probability a user remains active 30\u202fdays later). | `ℓ_ret(θ)` – a logistic regression on a retention label, or a survival‑analysis loss. |\n\nAll three objectives are **mutually compatible but often conflict**

In [33]:
# It's nice to know how to use "zip"
for competitor, answer in zip(competitors, answers):
    print(f"Competitor: {competitor}\n\n{answer}")


Competitor: openai/gpt-oss-20b

Below is a compact yet complete recipe you can drop into a production notebook, a slide deck, or a research paper.

---

## 1.  What you’re trying to optimise

| Target | What it means for a “good” recommender | Typical loss/metric |
|--------|----------------------------------------|---------------------|
| **Personalisation accuracy** | The model’s predictions should be close to a user’s true preference. | `ℓ_acc(θ)` – e.g. cross‑entropy or BPR‑rank loss. |
| **Fairness (demographic parity)** | The *average* probability of showing a recommended item should be the same for all protected groups. | `Δ_grp(θ) = |E[p | A=0] – E[p | A=1]|`. |
| **User retention** | Long‑term engagement (e.g. probability a user remains active 30 days later). | `ℓ_ret(θ)` – a logistic regression on a retention label, or a survival‑analysis loss. |

All three objectives are **mutually compatible but often conflict** (e.g. a pure accuracy‑maximising model can give a higher hit‑r

In [34]:
# Let's bring this together - note the use of "enumerate"

together = ""
for index, answer in enumerate(answers):
    together += f"# Response from competitor {index+1}\n\n"
    together += answer + "\n\n"

In [35]:
print(together)

# Response from competitor 1

Below is a compact yet complete recipe you can drop into a production notebook, a slide deck, or a research paper.

---

## 1.  What you’re trying to optimise

| Target | What it means for a “good” recommender | Typical loss/metric |
|--------|----------------------------------------|---------------------|
| **Personalisation accuracy** | The model’s predictions should be close to a user’s true preference. | `ℓ_acc(θ)` – e.g. cross‑entropy or BPR‑rank loss. |
| **Fairness (demographic parity)** | The *average* probability of showing a recommended item should be the same for all protected groups. | `Δ_grp(θ) = |E[p | A=0] – E[p | A=1]|`. |
| **User retention** | Long‑term engagement (e.g. probability a user remains active 30 days later). | `ℓ_ret(θ)` – a logistic regression on a retention label, or a survival‑analysis loss. |

All three objectives are **mutually compatible but often conflict** (e.g. a pure accuracy‑maximising model can give a higher hit‑rat

In [36]:
judge = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}}

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""


In [37]:
print(judge)

You are judging a competition between 4 competitors.
Each model has been given this question:

## 1.  Problem framing

We want a **streaming recommendation engine** that serves *millions* of users, each time step \(t\) we observe a user–context pair \((u_t, x_t)\), choose an item \(a_t\), see the feedback \(y_t \in \{0,1\}\) (e.g. click, view, purchase) and an *engagement* signal (e.g. dwell‑time, return‑rate).  
The system must

| Goal | How it is measured |
|------|--------------------|
| **Personalization accuracy** | Prediction error on the user‑specific preference model (e.g. negative log‑likelihood, squared error). |
| **Fairness / demographic parity** | The probability that a protected group \(g \in \mathcal{G}\) receives a positive recommendation should be the same (or within a tolerance \(\epsilon\)). |
| **User retention / engagement** | Long‑term reward (e.g. cumulative click‑through‑rate, session length, churn probability). |

We need a single **multi‑objective optimisation

In [38]:
judge_messages = [{"role": "user", "content": judge}]

In [42]:
# Judgement time!

groq = Groq(api_key=groq_api_key)
response = groq.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=judge_messages,
)
results = response.choices[0].message.content
print(results)


{"results":["1","3","4","2"]}


In [43]:
# OK let's turn this into results!

results_dict = json.loads(results)
ranks = results_dict["results"]
for index, result in enumerate(ranks):
    competitor = competitors[int(result)-1]
    print(f"Rank {index+1}: {competitor}")

Rank 1: openai/gpt-oss-20b
Rank 2: openai/gpt-oss-120b
Rank 3: llama3
Rank 4: mistral:instruct


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Which pattern(s) did this use? Try updating this to add another Agentic design pattern.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">These kinds of patterns - to send a task to multiple models, and evaluate results,
            are common where you need to improve the quality of your LLM response. This approach can be universally applied
            to business projects where accuracy is critical.
            </span>
        </td>
    </tr>
</table>